In [26]:
import os
from pyspark.sql import SparkSession

base_dir = r"E:\BIGDATA\data\parquet"
temp_dir = os.path.join(base_dir, "spark_temp")
os.makedirs(temp_dir, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("cic_clean_write")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.local.dir", temp_dir)
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
spark

In [27]:
parquet_path = r"E:\BIGDATA\data\parquet\cic_ddos_2019_10gb.parquet"

df_raw = spark.read.parquet(parquet_path)
df_raw.printSchema()

root
 |-- Unnamed: 0: integer (nullable = true)
 |-- Flow ID: string (nullable = true)
 |--  Source IP: string (nullable = true)
 |--  Source Port: integer (nullable = true)
 |--  Destination IP: string (nullable = true)
 |--  Destination Port: integer (nullable = true)
 |--  Protocol: integer (nullable = true)
 |--  Timestamp: timestamp (nullable = true)
 |--  Flow Duration: integer (nullable = true)
 |--  Total Fwd Packets: integer (nullable = true)
 |--  Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: double (nullable = true)
 |--  Total Length of Bwd Packets: double (nullable = true)
 |--  Fwd Packet Length Max: double (nullable = true)
 |--  Fwd Packet Length Min: double (nullable = true)
 |--  Fwd Packet Length Mean: double (nullable = true)
 |--  Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: double (nullable = true)
 |--  Bwd Packet Length Min: double (nullable = true)
 |--  Bwd Packet Length Mean: double (nullabl

# chuẩn hóa tên cột

In [28]:
from pyspark.sql import functions as F
import re

def norm(c):
    c = c.strip()
    c = re.sub(r"\s+", " ", c).replace("/", " per ")
    c = re.sub(r"[^0-9a-zA-Z ]+", " ", c)
    c = re.sub(r"\s+", "_", c).lower()
    return c.strip("_")

df = df_raw
for c in df_raw.columns:
    df = df.withColumnRenamed(c, norm(c))


print("Rows:", df.count())
df.printSchema()
df.show(5, truncate=False)

Rows: 24207109
root
 |-- unnamed_0: integer (nullable = true)
 |-- flow_id: string (nullable = true)
 |-- source_ip: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- destination_ip: string (nullable = true)
 |-- destination_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_duration: integer (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- total_backward_packets: integer (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable 

# Kiểm tra label

In [29]:
from pyspark.sql import functions as F

df.select("label").distinct().orderBy("label").show(200, truncate=False)


+-------------+
|label        |
+-------------+
|BENIGN       |
|DrDoS_DNS    |
|DrDoS_LDAP   |
|DrDoS_MSSQL  |
|DrDoS_NTP    |
|DrDoS_NetBIOS|
|DrDoS_SNMP   |
|Syn          |
|UDP-lag      |
|WebDDoS      |
+-------------+



# loại bỏ cột ko xác định

In [30]:
drop_cols = [c for c in ["unnamed_0", "flow_id"] if c in df.columns]
if drop_cols:
    df = df.drop(*drop_cols)

df.columns


['source_ip',
 'source_port',
 'destination_ip',
 'destination_port',
 'protocol',
 'timestamp',
 'flow_duration',
 'total_fwd_packets',
 'total_backward_packets',
 'total_length_of_fwd_packets',
 'total_length_of_bwd_packets',
 'fwd_packet_length_max',
 'fwd_packet_length_min',
 'fwd_packet_length_mean',
 'fwd_packet_length_std',
 'bwd_packet_length_max',
 'bwd_packet_length_min',
 'bwd_packet_length_mean',
 'bwd_packet_length_std',
 'flow_bytes_per_s',
 'flow_packets_per_s',
 'flow_iat_mean',
 'flow_iat_std',
 'flow_iat_max',
 'flow_iat_min',
 'fwd_iat_total',
 'fwd_iat_mean',
 'fwd_iat_std',
 'fwd_iat_max',
 'fwd_iat_min',
 'bwd_iat_total',
 'bwd_iat_mean',
 'bwd_iat_std',
 'bwd_iat_max',
 'bwd_iat_min',
 'fwd_psh_flags',
 'bwd_psh_flags',
 'fwd_urg_flags',
 'bwd_urg_flags',
 'fwd_header_length',
 'bwd_header_length',
 'fwd_packets_per_s',
 'bwd_packets_per_s',
 'min_packet_length',
 'max_packet_length',
 'packet_length_mean',
 'packet_length_std',
 'packet_length_variance',
 'fin_f

# ép kểu cột

In [31]:
from pyspark.sql.types import DoubleType, LongType, IntegerType

if "timestamp" in df.columns:
    df = df.withColumn("timestamp", F.to_timestamp("timestamp"))

int_like = [
    "source_port", "destination_port", "protocol",
    "total_fwd_packets", "total_backward_packets",
    "fwd_psh_flags", "bwd_psh_flags",
    "fwd_urg_flags", "bwd_urg_flags",
    "fin_flag_count", "syn_flag_count", "rst_flag_count",
    "psh_flag_count", "ack_flag_count", "urg_flag_count",
    "cwe_flag_count", "ece_flag_count",
    "inbound"
]

long_like = ["flow_duration", "fwd_header_length", "fwd_header_length_1"]

for c in int_like:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))

for c in long_like:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(LongType()))

for c, t in df.dtypes:
    if c not in (["label", "source_ip", "destination_ip", "simillarhttp", "timestamp"]):
        if t not in ("int", "bigint", "double", "float"):
            df = df.withColumn(c, F.col(c).cast(DoubleType()))
df.printSchema()

root
 |-- source_ip: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- destination_ip: string (nullable = true)
 |-- destination_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- total_backward_packets: integer (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nullable = true)
 |-- flow_bytes_per_s: double (nullabl

In [32]:
from pyspark.sql import functions as F

label_col = "label"

all_cols = df.columns
numeric_cols = [c for c, t in df.dtypes if c != label_col and t in ("int", "bigint", "double", "float", "long", "smallint", "tinyint")]

len(numeric_cols), numeric_cols[:10]


(81,
 ['source_port',
  'destination_port',
  'protocol',
  'flow_duration',
  'total_fwd_packets',
  'total_backward_packets',
  'total_length_of_fwd_packets',
  'total_length_of_bwd_packets',
  'fwd_packet_length_max',
  'fwd_packet_length_min'])

In [33]:
from pyspark.sql import functions as F

approx_stats = (
    df.select([F.approx_count_distinct(c).alias(c) for c in numeric_cols])
    .collect()[0]
    .asDict()
)

low_var_cols = [c for c in numeric_cols if approx_stats.get(c, 0) <= 1]

df = df.drop(*low_var_cols)
numeric_cols = [c for c in numeric_cols if c not in low_var_cols]

len(low_var_cols), len(numeric_cols)


(12, 69)

# loại bỏ cột ip

In [34]:
ip_cols = [c for c in ["source_ip", "destination_ip"] if c in df.columns]
if ip_cols:
    df = df.drop(*ip_cols)


In [35]:
len(df.columns)

72

# Lọai bỏ giá trị phi logic và null

In [36]:
from pyspark.sql import functions as F

conds = []

if "flow_duration" in df.columns:
    conds.append(F.col("flow_duration") < 0)

if "source_port" in df.columns:
    conds.append((F.col("source_port") < 0) | (F.col("source_port") > 65535))

if "destination_port" in df.columns:
    conds.append((F.col("destination_port") < 0) | (F.col("destination_port") > 65535))

rate_cols = [c for c in [
    "flow_bytes_per_s", "flow_packets_per_s",
    "fwd_packets_per_s", "bwd_packets_per_s"
] if c in df.columns]

for c in rate_cols:
    conds.append(F.col(c) < 0)


In [15]:
results = []

if "flow_duration" in df.columns:
    results.append(
        df.select(F.sum(F.when(F.col("flow_duration") < 0, 1).otherwise(0)).alias("viol"))
          .withColumn("rule", F.lit("flow_duration < 0"))
    )

if "source_port" in df.columns:
    results.append(
        df.select(F.sum(F.when((F.col("source_port") < 0) | (F.col("source_port") > 65535), 1).otherwise(0)).alias("viol"))
          .withColumn("rule", F.lit("source_port out of [0,65535]"))
    )

if "destination_port" in df.columns:
    results.append(
        df.select(F.sum(F.when((F.col("destination_port") < 0) | (F.col("destination_port") > 65535), 1).otherwise(0)).alias("viol"))
          .withColumn("rule", F.lit("destination_port out of [0,65535]"))
    )

for c in rate_cols:
    results.append(
        df.select(F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias("viol"))
          .withColumn("rule", F.lit(f"{c} < 0"))
    )

if results:
    out = results[0]
    for r in results[1:]:
        out = out.unionByName(r)
    out.select("rule", "viol").orderBy(F.desc("viol")).show(200, truncate=False)


+---------------------------------+----+
|rule                             |viol|
+---------------------------------+----+
|flow_duration < 0                |0   |
|source_port out of [0,65535]     |0   |
|destination_port out of [0,65535]|0   |
|flow_bytes_per_s < 0             |0   |
|flow_packets_per_s < 0           |0   |
|fwd_packets_per_s < 0            |0   |
|bwd_packets_per_s < 0            |0   |
+---------------------------------+----+



In [16]:
total = df.count()

if conds:
    any_violate = conds[0]
    for c in conds[1:]:
        any_violate = any_violate | c

    violate_rows = df.filter(any_violate).count()
    print("Total rows:", total)
    print("Rows violating at least one rule:", violate_rows)
    print("Violation ratio:", violate_rows / total if total else 0)
else:
    print("No rules applicable for this dataframe.")


Total rows: 24207109
Rows violating at least one rule: 0
Violation ratio: 0.0


In [13]:
INF = float("inf")

agg_exprs = []
for c in numeric_cols:
    col = F.col(c).cast("double")
    
    agg_exprs.append(
        F.sum(F.when(col.isNull(), 1).otherwise(0)).alias(f"{c}__null")
    )
    agg_exprs.append(
        F.sum(F.when(col.isNotNull() & F.isnan(col), 1).otherwise(0)).alias(f"{c}__nan")
    )
    agg_exprs.append(
        F.sum(F.when(col.isNotNull() & (F.abs(col) == F.lit(INF)), 1).otherwise(0)).alias(f"{c}__inf")
    )

bad_wide = df.groupBy(label_col).agg(*agg_exprs)
stack_items = []
for c in numeric_cols:
    stack_items.append(f"'{c}', `{c}__null`, `{c}__nan`, `{c}__inf`")

stack_sql = ", ".join(stack_items)

bad_long = bad_wide.selectExpr(
    label_col,
    f"stack({len(numeric_cols)}, {stack_sql}) as (col, null_count, nan_count, inf_count)"
)

bad_long = (
    bad_long
    .filter((F.col("null_count") > 0) | (F.col("nan_count") > 0) | (F.col("inf_count") > 0))
    .orderBy(F.desc("inf_count"), F.desc("nan_count"), F.desc("null_count"))
)

bad_long.show(200, truncate=False)


+-------------+------------------+----------+---------+---------+
|label        |col               |null_count|nan_count|inf_count|
+-------------+------------------+----------+---------+---------+
|Syn          |flow_packets_per_s|0         |0        |202306   |
|DrDoS_DNS    |flow_packets_per_s|0         |0        |162346   |
|DrDoS_DNS    |flow_bytes_per_s  |9         |0        |162337   |
|DrDoS_NetBIOS|flow_packets_per_s|0         |0        |129833   |
|DrDoS_NetBIOS|flow_bytes_per_s  |6         |0        |129827   |
|DrDoS_MSSQL  |flow_packets_per_s|0         |0        |126446   |
|DrDoS_MSSQL  |flow_bytes_per_s  |3         |0        |126443   |
|DrDoS_LDAP   |flow_packets_per_s|0         |0        |38630    |
|DrDoS_LDAP   |flow_bytes_per_s  |2         |0        |38628    |
|UDP-lag      |flow_packets_per_s|0         |0        |36382    |
|DrDoS_SNMP   |flow_packets_per_s|0         |0        |10609    |
|DrDoS_SNMP   |flow_bytes_per_s  |7         |0        |10602    |
|DrDoS_NTP

In [20]:
from pyspark.sql import functions as F

label_col = "label"

label_counts = (
    df.groupBy(label_col)
      .count()
      .withColumnRenamed("count", "label_total")
)

label_counts.orderBy(F.desc("label_total")).show(50, truncate=False)


+-------------+-----------+
|label        |label_total|
+-------------+-----------+
|DrDoS_SNMP   |5159870    |
|DrDoS_DNS    |5071011    |
|DrDoS_MSSQL  |4522492    |
|DrDoS_NetBIOS|4093279    |
|DrDoS_LDAP   |2179930    |
|Syn          |1582289    |
|DrDoS_NTP    |1202642    |
|UDP-lag      |366461     |
|BENIGN       |28696      |
|WebDDoS      |439        |
+-------------+-----------+



In [21]:
bad_ratio = (
    bad_long.join(label_counts, on=label_col, how="left")
            .withColumn("null_ratio", F.col("null_count") / F.col("label_total"))
            .withColumn("nan_ratio",  F.col("nan_count")  / F.col("label_total"))
            .withColumn("inf_ratio",  F.col("inf_count")  / F.col("label_total"))
)

bad_ratio.orderBy(
    F.desc("inf_ratio"),
    F.desc("nan_ratio"),
    F.desc("null_ratio")
).show(200, truncate=False)


+-------------+------------------+----------+---------+---------+-----------+---------------------+---------+---------------------+
|label        |col               |null_count|nan_count|inf_count|label_total|null_ratio           |nan_ratio|inf_ratio            |
+-------------+------------------+----------+---------+---------+-----------+---------------------+---------+---------------------+
|Syn          |flow_packets_per_s|0         |0        |202306   |1582289    |0.0                  |0.0      |0.12785654200970872  |
|UDP-lag      |flow_packets_per_s|0         |0        |36382    |366461     |0.0                  |0.0      |0.09927932303846794  |
|DrDoS_DNS    |flow_packets_per_s|0         |0        |162346   |5071011    |0.0                  |0.0      |0.032014523336667974 |
|DrDoS_DNS    |flow_bytes_per_s  |9         |0        |162337   |5071011    |1.7747940203639866E-6|0.0      |0.03201274854264761  |
|DrDoS_NetBIOS|flow_packets_per_s|0         |0        |129833   |4093279    

# điên các gia tri null,nan,inf

In [37]:
label_col = "label"
rate_cols = [c for c in ["flow_bytes_per_s", "flow_packets_per_s"] if c in df.columns]

INF = float("inf")

for c in rate_cols:
    col = F.col(c).cast("double")
    df = df.withColumn(
        c,
        F.when(col.isNull(), F.lit(None))
         .when(F.isnan(col), F.lit(None))
         .when(F.abs(col) == F.lit(INF), F.lit(None))
         .otherwise(col)
    )


In [38]:
median_exprs = [
    F.percentile_approx(F.col(c).cast("double"), 0.5, 10000).alias(f"{c}__med")
    for c in rate_cols
]

med_by_label = df.groupBy(label_col).agg(*median_exprs)


In [39]:
df = df.join(med_by_label, on=label_col, how="left")

for c in rate_cols:
    df = df.withColumn(
        c,
        F.coalesce(F.col(c), F.col(f"{c}__med"))
    )


In [40]:
global_meds = df.agg(*[
    F.percentile_approx(F.col(c).cast("double"), 0.5, 10000).alias(f"{c}__gmed")
    for c in rate_cols
])

df = df.crossJoin(global_meds)

for c in rate_cols:
    df = df.withColumn(
        c,
        F.coalesce(F.col(c), F.col(f"{c}__gmed"))
    )


In [41]:
drop_cols = []
for c in rate_cols:
    drop_cols += [f"{c}__med", f"{c}__gmed"]

df = df.drop(*[c for c in drop_cols if c in df.columns])


In [22]:
bad_check = df.select([
    F.sum(
        F.when(
            F.col(c).isNull() | F.isnan(F.col(c).cast("double")) | (F.abs(F.col(c).cast("double")) == F.lit(INF)),
            1
        ).otherwise(0)
    ).alias(c)
    for c in rate_cols
])

bad_check.show(truncate=False)


+----------------+------------------+
|flow_bytes_per_s|flow_packets_per_s|
+----------------+------------------+
|0               |0                 |
+----------------+------------------+



In [42]:
# df = df.dropDuplicates()
print("Rows after clean:", df.count())


Rows after clean: 24207109


In [43]:
len(df.columns)

72

In [44]:
df_clean = df

output_path = "file:///E:/BIGDATA/data/parquet/cic_ddos_2019_10gb_cleaned.parquet"
df_clean.write.mode("overwrite").parquet(output_path)


In [16]:
spark.stop()